# Tumor-edge Communication Niche Simulation

Select **cccphe** and **Run All**. Default: 8,000 cells, one tissue, five types, 75 real LR pairs, two six-LR designed MIs, four evaluation-only niches. This is a synthetic downstream experiment, separate from clinical CCC/Cox and existing Xenium outputs. Full execution has not been performed by Codex.

Sources: [scCube](https://github.com/ZJUFanLab/scCube) expression VAE and Poisson spatial APIs; [SpiderNet simulation](https://github.com/ma-compbio-lab/SpiderNet/blob/c142960620f2fa584e8843c9acf48d719c1f2f8a/Tutorial/Simulation/SimulationData_generation.py) sender/receiver expression injection concept; [A meta-interaction basis for cell-cell communication in tissues](https://doi.org/10.64898/2026.09.21.753369). This implements a microenvironment benchmark, not a reproduction of the complete SpiderNet paper or edge encoder.

Default lambda_comp=0.005. Use config/tumor_edge_simulation_ccc_only.json for the matched CCC-only control. Previously uncalibrated results remain in their original NAS directory. This corrected simulation uses an independent output directory; it has passed miniature tests but its full 8,000-cell recovery has not been run or guaranteed.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys,json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT=Path('/home/xueshuailin/CCC_Phe')
sys.path.insert(0,str(ROOT/'src'))
from phenoniche.simulation.config import SimulationConfig
from phenoniche.simulation.dependencies import prepare_environment
from phenoniche.simulation.workflow import dataset,run_experiment
from phenoniche.simulation.adapter import adapt
from phenoniche.simulation.plotting import SimulationReport
from phenoniche.training import set_seed
config=SimulationConfig.from_json(ROOT/'config/tumor_edge_simulation.json')
set_seed(config.seed)
display(pd.Series(prepare_environment()).to_frame('Installed version'))
display(pd.Series(config.to_dict()).to_frame('Configuration'))
OUTPUT=Path(config.output_dir)
def show(fig):
    display(fig)
    plt.close(fig)

## 1. scCube expression/spatial generation and MI injection

scCube is trained on a self-contained synthetic negative-binomial reference with five type marker sets, not on hidden external patient data. Its generated log-normalized values are inverted and clipped. Explicit count-space seed-type mean calibration restores the intended cell-type expression differences while retaining within-type relative variation, then Poisson sampling is applied. This is a hybrid simulator, not an assertion of native VAE fidelity. Raw uncalibrated counts and before/after marker audits are saved. Calibration never uses niche or MI labels. scCube Poisson coordinates are transformed into uniform half-disc coordinates in micrometers and reflected with identical radial type assignments. Mirror donors remain in the same evaluation fold. The spatial pairing-only switch is off by default; in that setting entire receiver profiles are permuted within sample/type/radial strata to preserve conditional expression marginals.

MI-1 and MI-2 use the same Tumor → Fibroblast roles but gene-disjoint characteristic LR groups. Signal is injected into sender ligands and receiver receptors only, never into final CCC.

In [ ]:
adata,lr=dataset(config)
display(adata)
display(lr)
print('Source provenance:',adata.uns['simulation_provenance'])
display(pd.DataFrame(json.loads((OUTPUT/'expression_fidelity.json').read_text())['raw_scCube']))
display(pd.DataFrame(json.loads((OUTPUT/'expression_fidelity.json').read_text())['final_counts']))

## 2. Exact directed CCC and evaluation-only ground truth

KNN=30, sigma=20 µm, and the existing physical-pair/opportunity-normalized CCC implementation are reused. Ground truth, designed molecular loadings and pair-weighted MI exposure are saved independently; the model adapter only passes expression, geometry, CCC and composition supervision. For a single tissue, mirrored angular sectors span the full radius and define folds using geometry only; sample folds are used for multiple replicates. Shared physical cells are purged. True labels are never used to assign or choose folds. Composition matching is checked locally, by radial strata, and by a diagnostic grouped-CV composition-only AUC.

In [ ]:
data=adapt(adata,lr,config,OUTPUT)
print('CCC shape:',data.targets.shape,'effective training features:',int(data.feature_mask.sum()))
print('Train/validation/test centers:',[len(r) for r in data.rows])
print('Composition shape:',data.composition_target.shape)
truth_report=SimulationReport(OUTPUT,load_results=False)
show(truth_report.ground_truth())

## 3. Train current GNN, discover niches, and run all controls

The same expression-only GNN learns nonnegative 32-program Z, H, and composition B. No cell-type embedding or regulatory alignment is added. CCC-vector AE reuses the existing three-stage training code. Composition clustering consumes known type fractions; expression PCA uses label-free pair-weighted neighborhood expression; AE consumes precomputed CCC; GNN predicts CCC from expression/geometry with auxiliary composition supervision. These are different information conditions. The composition dictionary is initialized from training-only CCC prototypes and their observed composition, without any true niches or MI labels; the model architecture and loss are unchanged.

GNN K=4–15 is chosen using training/validation data, never true niches. The main figure separately uses fixed K=4 for all methods. Matching cluster labels to truth is performed only for evaluation, fit on training labels and then applied to held-out test centers. Raw clusters are independently saved and shown.

In [ ]:
run_experiment(config)
report=SimulationReport(OUTPUT)
show(report.reconstruction())
display(pd.read_csv(OUTPUT/'gnn/K_selection.csv'))
print('Raw unaligned clusters:')
display(pd.read_csv(OUTPUT/'raw_clusters.csv').head())

## 4. Main comparison: Ground Truth | Composition | Expression | CCC-AE | CCC_Phe GNN

These are the actual raw cluster outputs. Colors are independently assigned per method and are not post-hoc aligned to the ground truth. The full single-tissue map is descriptive/transductive; held-out metrics below use disjoint physical cells. For multiple replicates, the figure displays a held-out replicate.

In [ ]:
show(report.main_figure())
show(report.adaptive_niches())

## 5. Quantitative recovery and composition controls

All-cell/transductive and held-out results are reported separately. ARI/NMI and edge-only ARI are label permutation invariant. Edge F1/IoU and A/B accuracy use training-only evaluation mappings. The composition diagnostic classifier uses true A/B labels only here, with paired donors grouped in CV; it is not part of the niche model. Results may show failure to distinguish the two half-rings. One synthetic realization does not establish biological causality or multi-seed robustness.

In [ ]:
display(pd.read_csv(OUTPUT/'metrics.csv'))
display(pd.read_csv(OUTPUT/'evaluation_split_coverage.csv'))
display(pd.Series(json.loads((OUTPUT/'diagnostics.json').read_text())).to_frame('Diagnostic'))
show(report.composition_audit())
display(pd.read_csv(OUTPUT/'radial_composition_audit.csv'))

## 6. True MI-1 / MI-2 versus recovered communication programs

Two distinct learned programs are matched by their H dictionary cosine with the designed MI-LR dictionary, without using held-out activity to choose programs. Both overall and edge-restricted test activity correlations are reported. Initial H similarity is reported separately because H uses unsupervised CCC prototype initialization; a recognizable H alone does not show the encoder learned MI activity. Partial or poor dictionary/activity recovery is shown as measured; learned programs need not equal the two designed MIs one-to-one.

In [ ]:
display(pd.read_csv(OUTPUT/'MI_matches.csv'))
show(report.recovered_programs())
show(report.dictionary())
display(pd.read_csv(OUTPUT/'gnn/programs/top_ccc.csv').groupby('program').head(5))
print('All generated data, checkpoints, CSVs, PNG/PDF:',OUTPUT)